### Extract WHO mutations

This is the script to transfer from VCF file to CSV file for SNP mutations.

Actually, WHO provided an Excel file with better pre-processed information can be used directly. 

In [ ]:
import os

path = "/Users/sijianfan/Documents/projects/BiSSGL/datasets/realAnalysis/tuberculosis/mutation-catalogue-2023/Final Result Files"
filename = "Genomic_coordinates_7May2024.vcf.gz"

In [ ]:
import pandas as pd
from cyvcf2 import VCF
import re

vcf_fn = f"{path}/{filename}"  # <- 改成 repo 中实际的 vcf 文件名
out_fn = f"{path}/who_mutations.csv"

vcf = VCF(vcf_fn)
rows = []
for rec in vcf:
    pos = rec.POS
    ref = rec.REF
    # consider multiallele: take each ALT separately
    for alt in rec.ALT:
        # extract variant name and gene/drug from INFO or ID fields if present
        # the GTB repo VCF usually maps variant names into ID or INFO fields.
        mut_name = rec.ID if rec.ID else f"{rec.CHROM}_{pos}_{ref}>{alt}"
        info = rec.INFO
        # try to get gene/drug from INFO (depends on file format)
        gene = info.get("GENE") if "GENE" in info else ""
        drug = info.get("DRUG") if "DRUG" in info else ""
        notes = ""
        rows.append(
            {
                "mut_name": mut_name,
                "gene": gene,
                "pos": pos,
                "ref": ref,
                "alt": alt,
                "drug": drug,
                "notes": notes,
            }
        )

df = pd.DataFrame(rows)
df.to_csv(out_fn, index=False)
print("Wrote", out_fn, "with", len(df), "rows")

Wrote /Users/sijianfan/Documents/projects/BiSSGL/datasets/realAnalysis/tuberculosis/mutation-catalogue-2023/Final Result Files/who_mutations.csv with 118431 rows
